# Lesson 03: AI Evaluation Fundamentals — How to Judge AI Responses

## Learning Objectives
- Understand why evaluating AI output is much harder than evaluating traditional software
- Design simple scoring criteria and manually evaluate AI responses
- Use "AI as a Judge" to evaluate AI
- Critically examine AI output and develop an evaluation mindset

> Evaluation is the most important and most often overlooked part of AI Engineering. Without evaluation, you have no idea whether your AI is improving or degrading.

## Environment Setup

> Please run `00_Environment_Setup.ipynb` first to set up dependencies and API keys,
> then return to this notebook.

Once done, run the cell below to load environment variables:

In [ ]:
# Load API key from .env file (no need to enter it every time)
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# ===== Pick a provider: change this one line, nothing else =====
#   'openai'     cloud  needs OPENAI_API_KEY      strongest, has embeddings
#   'deepseek'   cloud  needs DEEPSEEK_API_KEY    cheapest cloud, no embeddings
#   'openrouter' cloud  needs OPENROUTER_API_KEY  many vendors, no embeddings
#   'ollama'     local  no key, free and offline  run `ollama serve` and pull the model first
PROVIDER = 'openai'

# All four speak the OpenAI API format. They differ only in URL, key, model names.
PROVIDERS = {
    'openai': {
        'base_url': None,                            # None = OpenAI's default endpoint
        'api_key': os.getenv('OPENAI_API_KEY'),
        'model': 'gpt-5.6-luna',                     # small model: cheap and fast
        'model_big': 'gpt-5.6-terra',                # big model: pricier and stronger
        'embedding_model': 'text-embedding-3-small',
    },
    'deepseek': {
        'base_url': 'https://api.deepseek.com/v1',
        'api_key': os.getenv('DEEPSEEK_API_KEY'),
        'model': 'deepseek-v4-flash',                # fast and cheap
        'model_big': 'deepseek-v4-pro',              # stronger and slower; both V4 models think first
        'embedding_model': None,                     # DeepSeek has no embeddings endpoint
    },
    'openrouter': {
        'base_url': 'https://openrouter.ai/api/v1',
        'api_key': os.getenv('OPENROUTER_API_KEY'),
        'model': 'openai/gpt-5.6-luna',
        'model_big': 'openai/gpt-5.6-terra',
        'embedding_model': None,                     # OpenRouter does not proxy embeddings
    },
    'ollama': {
        'base_url': os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434/v1'),
        'api_key': 'ollama',                         # local models ignore the key
        'model': 'gemma4:e2b-mlx',                   # run: ollama pull gemma4:e2b-mlx
        'model_big': 'gemma4:e2b-mlx',
        'embedding_model': 'nomic-embed-text',       # run: ollama pull nomic-embed-text
    },
}

cfg = PROVIDERS[PROVIDER]

# Check the key first: with no key, the OpenAI client raises a long traceback.
if not cfg['api_key']:
    raise SystemExit(
        f"No API key for '{PROVIDER}'. Either add {PROVIDER.upper()}_API_KEY to your .env file,\n"
        f"or set PROVIDER = 'ollama' above to run locally with no key at all."
    )

client = OpenAI(api_key=cfg['api_key'], base_url=cfg['base_url'])

# Every lesson below uses only these three names, so switching provider needs no
# other code change.
MODEL = cfg['model']
MODEL_BIG = cfg['model_big']
EMBEDDING_MODEL = cfg['embedding_model']

print(f'Connected! provider = {PROVIDER}, default model = {MODEL}')


---

## Activity 1: Be Your Own "AI Judge" — Manual Evaluation

### Activity Goal
First, have AI answer several questions. Then design your own scoring criteria and score each response item by item. Build your judgment of "what makes a good answer."

In [ ]:
# Three questions from different domains
questions = [
    {'category': 'General Knowledge', 'question': 'Explain the basic principle of the greenhouse effect and its impact on Earth\'s climate.'},
    {'category': 'Creative', 'question': 'Write a 200-word sci-fi story opening. Theme: the last human talks to an AI.'},
    {'category': 'Logic', 'question': 'If all cats are afraid of water, and Xiao Ming\'s pet is not afraid of water, is Xiao Ming\'s pet a cat? Analyze the reasoning process.'}
]

answers = {}
for i, q in enumerate(questions):
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{'role':'user','content':q['question']}],
        temperature=0.5)
    answers[i] = r.choices[0].message.content
    print(f'\nQuestion {i+1} [{q["category"]}]')
    print(f'Q: {q["question"]}')
    print(f'AI Response:\n{answers[i]}')
    print()

### Manual Evaluation Task

Now it's your turn to be the "AI judge". Score each response on these criteria (1-5 each):

| Dimension | 1 pt | 3 pts | 5 pts |
|-----------|------|-------|-------|
| Accuracy | Has factual errors | Mostly correct, minor flaws | Fully accurate |
| Completeness | Missing key info | Covers most points | Comprehensive and thorough |
| Clarity | Confusing, hard to follow | Generally coherent | Clear, well-structured |
| Usefulness | Not helpful at all | Somewhat helpful | Directly solves the problem |

Fill in your scores in the code cell below:

In [ ]:
# Enter your scores here (1-5 for each dimension)
# Four dimensions: Accuracy, Completeness, Clarity, Usefulness
my_scores = {
    1: [4, 4, 5, 4],  # Question 1: adjust based on actual output
    2: [3, 4, 4, 3],  # Question 2
    3: [5, 3, 4, 4],  # Question 3
}

print('Your evaluation results:')
print(f'{"Question":<10}{"Accuracy":<14}{"Completeness":<14}{"Clarity":<14}{"Usefulness":<14}{"Total":<8}')
print('-' * 74)
for q_num, scores in my_scores.items():
    total = sum(scores)
    print(f'{"Q" + str(q_num):<10}{scores[0]:<14}{scores[1]:<14}{scores[2]:<14}{scores[3]:<14}{total:<8}')
best = max(my_scores, key=lambda k: sum(my_scores[k]))
print(f'\nHighest score: Question {best}, total {sum(my_scores[best])}')

### Discussion
- When scoring AI responses, which dimension was hardest to judge?
- If your scores differed from a classmate's, where was the divergence?
- What characteristics should a good evaluation rubric have?

---

## Activity 2: Using AI as a Judge — "AI as a Judge"

### Activity Goal
Let AI evaluate AI responses. Give AI a scoring rubric and let it score. Then compare AI's scores with your own.

In [ ]:
# Let AI act as evaluator
eval_prompt = ('You are a strict AI output evaluation expert. Rate AI responses according to the following criteria.\n'
'Scoring criteria (1-5 each):\n'
'1. Accuracy: 1=clear errors, 5=completely accurate\n'
'2. Completeness: 1=serious omissions, 5=comprehensive\n'
'3. Clarity: 1=confusing, 5=clear and rigorous\n'
'4. Usefulness: 1=not helpful, 5=very helpful\n'
'Rate each response item by item.')

for i, q in enumerate(questions):
    print(f'\n=== AI Judge Evaluation: Question {i+1} [{q["category"]}] ===')
    r = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role':'system','content':'You are a strict AI output evaluation expert.'},
            {'role':'user','content':f'{eval_prompt}\n\nQuestion: {q["question"]}\n\nAI Response: {answers[i]}'}
        ],
        temperature=0.2)
    print(r.choices[0].message.content)

### Discussion
- Do the AI judge's scores match yours?
- Does the AI judge tend to be "soft" (biased toward high scores)?
- If you rephrase the scoring criteria, would the AI's scores change?

---

## Activity 3: A/B Comparison — Which AI Writes Better?

### Activity Goal
Don't look at scores — just compare which is better. Put two responses side by side and compare directly — this is the most intuitive evaluation method.

In [ ]:
# A/B blind comparison
topic = 'The impact of AI on education'

# Response A: Academic style
print('[Response A: Academic Style]')
ra = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role':'system','content':'You are an education researcher. Write in a rigorous academic style.'},
        {'role':'user','content':f'Write a 200-word short essay on {topic}.'}
    ], temperature=0.3)
text_a = ra.choices[0].message.content
print(text_a)

# Response B: Casual style
print('\n[Response B: Casual Style]')
rb = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role':'system','content':'You are a popular science blogger. Write in an engaging, easy-to-understand style.'},
        {'role':'user','content':f'Write a 200-word short essay on {topic}.'}
    ], temperature=0.7)
text_b = rb.choices[0].message.content
print(text_b)

# AI blind evaluation
print('\n[AI Blind Evaluation]')
rj = client.chat.completions.create(
    model=MODEL,
    messages=[{'role':'user','content':f'Compare these two short essays and determine which is better with reasons.\n\nEssay A: {text_a}\n\nEssay B: {text_b}'}],
    temperature=0.2)
print(rj.choices[0].message.content)

### Discussion
- Do you agree with the AI judge's blind evaluation result?
- Academic style vs casual style: which suits the topic of "education" better?
- If you were choosing a content creation assistant, which style would you prefer?

---

## Lesson Review

| Skill | Description |
|-------|-------------|
| Manual evaluation | Design scoring criteria and personally score AI responses |
| AI as judge | Use AI to evaluate AI; understand its strengths and limitations |
| Rubric design | Experience how the wording of criteria affects evaluation results |
| A/B comparison | Directly compare two responses to judge which is better |

### Homework
1. Find 5 AI responses from different domains and score them using the same criteria
2. Design your own evaluation rubric with at least 5 dimensions
3. Think: if you were to build an automated evaluation system, how would you design it?